# Fine-tune a Hebrew NLI model on clean HebNLI  ·  Person B

`nli_rerank.py` defaults to `oriel9p/AlephBERT-FT-HebNLI-LCHAIM`, which was fine-tuned
on all of HebNLI — including the rows the probe was mined from. It has already seen our
(target, negation) pairs labelled `contradiction`, so its probe scores are partly recall.
This notebook builds the replacement.

**Runs in two places.** The Colab web UI, and VS Code with the Colab extension. In both
the kernel is a remote Google VM — your local files are *not* there, which is why step 1
clones the repo. Cells 1–7 are CPU-only; the GPU is needed from step 8.

Where the two environments differ is Colab's frontend features — stored secrets, Drive
mounting, browser downloads. Every cell below falls back rather than failing.

## 1. Setup

In [ ]:
# Secrets, three ways: Colab's stored secrets, then the environment, then a prompt.
# The VS Code extension has no access to Colab's secret store, so the fallbacks are
# what make this notebook portable. getpass keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

In [ ]:
# What are we actually running on? Answers 'will this work here' before anything slow.
import sys, platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')
subprocess.run(['git','pull','-q','origin',BRANCH], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin',
                f'https://github.com/{OWNER}/{REPO}.git'], check=True)
del gh_token, url

!pip install -q -r requirements.txt
!git log --oneline -3

In [ ]:
# offline checks first — seconds, no network, and everything downstream inherits
# any fault they would have caught
!python -m src.data.negation --selftest
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_projection | tail -3
!python -m tests.test_nli_data | tail -3

## 2. Data

HebNLI's repo card marks it private, so the download needs a token. `src/data/hebnli.py`
reads `HF_TOKEN` from the environment.

Already run once locally, with these results — the cells below should reproduce them:

| split | loaded | promptID filter | text audit | kept |
|---|---|---|---|---|
| train | 300,067 | −2,068 | −9 | 297,990 |
| val | 1,999 | −9 | −1 | 1,989 |
| test | 884 | −1 | 0 | 883 |

In [ ]:
os.environ['HF_TOKEN'] = get_secret('HF_TOKEN')

In [ ]:
!python -m src.data.hebnli --split train --out data/raw/hebnli_train.jsonl
!python -m src.data.hebnli --split val   --out data/raw/hebnli_val.jsonl
!python -m src.data.hebnli --split test  --out data/raw/hebnli_test.jsonl

In [ ]:
# Two filters. The promptID list is Itay's 689 held-out prompts; the text audit catches
# probe sentences reachable under a *different* promptID, which an id filter cannot see.
# All three splits — val shares prompts with train, so an unfiltered val would measure
# validation accuracy on rows the probe came from.
# One line each, no loop: IPython rewrites `!` magics line by line, so a backslash
# continuation inside a for-body is parsed as Python and fails.
!python -m src.nli.prepare_data --source data/raw/hebnli_train.jsonl --split train --out data/raw/hebnli_train_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_val.jsonl   --split val   --out data/raw/hebnli_val_clean.jsonl
!python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl  --split test  --out data/raw/hebnli_test_clean.jsonl

In [ ]:
import json
for split in ('train', 'val', 'test'):
    m = json.load(open(f'results/nli_data_{split}.json', encoding='utf-8'))
    f = m['funnel']
    print(f"{split:6s} loaded={f['loaded']:>7}  id_filter=-{f['loaded']-f['prompt_id_clean']:<5}"
          f"  text_audit=-{m['text_overlap']['rows_dropped']:<3}  kept={m['rows_written']}")

## 3. Smoke run  ·  GPU from here

2000 rows, one epoch, a few minutes. Exercises tokenising, the label map, the training
loop, checkpoint saving and the manifest before hours are committed to any of it.
Deliberately on the VM's own disk with no epoch checkpoints — the weights are throwaway.

Its accuracy is recorded as `smoke_run` and must not be reported.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --max-train 2000 --epochs 1

## 4. Full run

Hours over ~298k rows. Two things decide whether that survives:

**Where the weights go.** `/content` is wiped when the runtime resets. Drive is the only
durable option, and Drive mounting is a Colab-frontend feature — it may not work under the
VS Code extension. The next cell tries, reports honestly, and picks a path either way.

**`--save-epochs`.** Writes a resumable checkpoint per epoch, keeping the two newest.
Without it nothing exists on disk until training completes. If the session dies, re-run
the training cell unchanged — it resumes from the newest checkpoint. `--fresh` starts over.

In [ ]:
# Drive if we can get it, VM disk if we cannot — but say which, loudly, because the
# difference is whether a disconnect costs an epoch or the whole run.
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_DRIVE = True
except Exception as exc:
    ON_DRIVE = False
    CKPT = 'checkpoints/alephbert-hebnli-clean'
    print(f'[warn] Drive unavailable ({type(exc).__name__}: {exc})')
    print('[warn] checkpoints go to the VM disk and die with the runtime.')
    print('[warn] epoch checkpoints still protect against a crashed cell, not a reset.')
    print('[warn] plan to push the finished model somewhere before the session ends.')
print('checkpoint dir:', CKPT, '| durable:', ON_DRIVE)

In [ ]:
# re-run this exact cell after a disconnect - it resumes from the newest checkpoint
!python -m src.nli.train_nli --base alephbert \
    --train data/raw/hebnli_train_clean.jsonl \
    --val data/raw/hebnli_val_clean.jsonl \
    --save-epochs --out {CKPT}

## 5. Verify and keep the results

`check_nli_labels` runs six obvious Hebrew pairs and prints the names from
`config.id2label` alongside what the model predicts. For the released checkpoint this
discovers an undocumented mapping; for ours it confirms the names we wrote survived
training and describe what the model does — a config can say anything.

`results/*.csv` and `*.json` are a few KB and belong in git so both of us see the same
numbers. Weights do not — `.gitignore` blocks them.

In [ ]:
!python -m src.interventions.check_nli_labels \
    --model {CKPT} --subfolder ""

In [ ]:
import pandas as pd
pd.read_csv('results/nli_train.csv')

In [ ]:
# Browser download is Colab-frontend only. Under VS Code, print the contents instead
# so the numbers can be copied straight out of the saved notebook.
RESULTS = ['results/nli_train.csv', 'results/nli_train_alephbert.json',
           'results/nli_data_train.json', 'results/nli_data_val.json',
           'results/nli_data_test.json']
try:
    from google.colab import files
    for path in RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__}) — contents below\n')
    for path in RESULTS:
        print(f'===== {path} =====')
        print(open(path, encoding='utf-8').read())

Commit the result files from your machine with the `nli:` prefix.

**Still open after this notebook:** `nli_rerank.py` now has a `pair` encoding mode and
reads labels from `config.id2label`, so it *can* load this checkpoint — but `lam` still
defaults to 1.0, meaning pure NLI with the embedder's cosine contributing nothing. Tuning
λ on the probe's train split is the next piece.